# SAFE binary v6 — 외부 데이터셋(DKTC·K-MHaS·APEACH) 반복 재학습 + fresh blind v8

blind v1~v7은 모두 소비됐으므로 **개발 회귀셋**으로만 쓰고, 새 **blind v8**은 후보를 고정한 뒤 정확히 한 번만 평가한다(해시 검증). blind 원문은 학습에 절대 병합하지 않는다.

- **신규 소스(R1 한국어 즉시 트랙, 번역 불필요)**:
  - **DKTC** (협박·갈취·직장내/기타 괴롭힘, 전량 `주의`) — GitHub `tunib-ai/DKTC`, CC-BY-NC-SA
  - **K-MHaS** (멀티라벨 혐오 → 이진 붕괴, 정상/주의 혼재) — HF `jeanlee/kmhas_korean_hate_speech`
  - **APEACH** (혐오 이진) — HF `jason9693/APEACH`
  - 셋 다 `scripts/adapt_external_datasets.py`로 홀드아웃(aihub/beep)+소비 blind와 dedup 후 `data/synthetic/{dktc,kmhas,apeach}.jsonl` 생성. config = `configs/module1_binary_hardcases_v6.yaml`.
- **타깃**: 협박·갈취(digital_extortion)·괴롭힘 슬라이스 강화. 기존 슬라이스(fraud·coercive·grooming) 회귀 금지.
- **분포 리스크(정직 고지)**: DKTC가 전량 주의(15k)라 신규 주입이 ~67% 주의로 치우침 → 합산 주의비율 상승. **specificity·warm_normal 과플래그를 dev 게이트로 반드시 감시**.
- **규칙 보조 OFF**: 운영 `SAFE_RULE_ASSIST=0` 이므로 dev·blind 평가 모두 모델 단독.
- **한계**: blind v8도 저자 생성 합성셋 → *패턴 일반화* 측정. 실데이터 회귀는 aihub/beep holdout으로 별도 확인.
- 실행은 Colab A100에서 순서대로. HF 업로드 셀은 기본 OFF.

In [ ]:
# 1. A100 및 저장소 확인
import subprocess, sys, json, hashlib, re, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
print(torch.cuda.get_device_name(0))
REPO=Path('/content/thisabled-ai')
BRANCH='feature/grooming-augmentation'
REMOTE='https://github.com/threeGuineas/thisabled-ai.git'
if REPO.exists():
    subprocess.run(['git','pull','--ff-only','origin',BRANCH],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
assert (REPO/'.git').exists(),f'저장소 준비 실패: {REPO}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','datasets'],cwd=REPO,check=True)  # 외부 어댑터용
sys.path.insert(0,str(REPO))

In [ ]:
# 2. data_bundle.zip 업로드 — VS Code/Jupyter widget
import io, zipfile, ipywidgets as widgets
from IPython.display import display
uploader=widgets.FileUpload(accept='.zip',multiple=False,description='data_bundle.zip 선택')
def on_upload(change):
    value=uploader.value
    if not value:return
    item=next(iter(value.values())) if isinstance(value,dict) else value[0]
    with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
        required={'data/eval/aihub_train.jsonl','data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl'}
        missing=sorted(required-set(z.namelist())); assert not missing,missing
        z.extractall(REPO)
    print('업로드 완료')
uploader.observe(on_upload,names='value'); display(uploader)

In [ ]:
# 3. 데이터 빌드 + 외부 데이터셋 어댑터 + 누수 가드 (dev=blind v1~v7 / fresh=blind v8)
required=[REPO/'data/eval/aihub_train.jsonl',REPO/'data/eval/aihub_real_holdout.jsonl',REPO/'data/eval/beep_real_holdout.jsonl']
assert all(p.exists() for p in required),[str(p) for p in required if not p.exists()]
# fresh blind v8은 사용자가 사전 작성(40행, 슬라이스/label/receiver_is_minor 스키마 = v5~v7과 동일)
BLIND_V8=REPO/'tests/fixtures/safe_blind_v8.jsonl'
assert BLIND_V8.exists(), 'fresh blind v8 없음 — tests/fixtures/safe_blind_v8.jsonl(40행)을 먼저 작성하라(방법론: 라운드당 새 blind 1개).'
steps=[
    [sys.executable,'scripts/download_seed_datasets.py'],
    [sys.executable,'scripts/build_processed_dataset.py'],
    [sys.executable,'scripts/build_final_dataset.py','--synth-repeat','1','--include-aihub-train'],
    # v6 config가 참조하는 v5 하드케이스 재생성 (forbidden = 소비 blind + fresh v8)
    [sys.executable,'scripts/build_safe_hardcase_dataset.py','--include-v5',
     '--output','data/synthetic/safe_hardcases_v5/train.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v6.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v7.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v8.jsonl'],
    # 외부 3소스 다운로드→어댑터→dedup (홀드아웃+blind v1~v8과 교차)
    [sys.executable,'scripts/adapt_external_datasets.py'],
]
for cmd in steps: subprocess.run(cmd,cwd=REPO,check=True)
import pandas as pd
from src.data.dedup import find_duplicate_indices
train=pd.read_parquet(REPO/'data/processed/train.parquet')
def norm(x):return re.sub(r'[^0-9a-z가-힣]+','',str(x).lower())
train_norm={norm(t) for t in train['text']}
# 신규 외부 소스 + 하드케이스 = 학습에 실제로 들어가는 extra
extra_paths=['data/synthetic/dktc.jsonl','data/synthetic/kmhas.jsonl','data/synthetic/apeach.jsonl','data/synthetic/safe_hardcases_v5/train.jsonl']
extra=[]
for p in extra_paths: extra+=[json.loads(x) for x in (REPO/p).read_text().splitlines() if x]
extra_texts=[x['text'] for x in extra]; extra_norm={norm(t) for t in extra_texts}
blindv8=[json.loads(x) for x in BLIND_V8.read_text().splitlines() if x]; bt=[x['text'] for x in blindv8]
# (1) 신규 extra ↔ 소비된 blind v1~v7 완전 중복 0
for i in range(1,8):
    name=f'safe_blind_v{i}.jsonl'
    blind=[json.loads(x) for x in (REPO/'tests/fixtures'/name).read_text().splitlines() if x]
    assert not ({norm(x['text']) for x in blind}&extra_norm), f'extra leak vs {name}'
# (2) fresh blind v8 ↔ train·extra 완전/근사(0.8) 중복 0
assert not ({norm(t) for t in bt}&extra_norm), 'blind v8 exact leak vs extra'
assert not ({norm(t) for t in bt}&train_norm), 'blind v8 exact leak vs train'
assert not find_duplicate_indices(extra_texts, bt, threshold=0.8), 'blind v8 near-dup vs extra'
assert not find_duplicate_indices(list(train['text']), bt, threshold=0.8), 'blind v8 near-dup vs train'
print('base train',len(train),'| extra(신규+하드케이스)',len(extra),
      '| extra label',pd.Series([x.get('label') for x in extra]).value_counts(dropna=False).to_dict())
print('blind v8 clean vs train/extra: OK')

In [ ]:
# 4. 개발 평가 함수 — dev 회귀셋 = 실데이터 holdout + 소비된 blind v1~v7. blind v8은 여기서 읽지 않는다. 규칙 보조 OFF.
import numpy as np, yaml
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer,AutoModelForSequenceClassification
real=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    real += [json.loads(x) for x in (REPO/'data/eval'/name).read_text().splitlines() if x]
dev=[]
for i in range(1,8):
    dev += [json.loads(x) for x in (REPO/'tests/fixtures'/f'safe_blind_v{i}.jsonl').read_text().splitlines() if x]
groom=[]
for split in ['val','test']: groom += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').read_text().splitlines() if x]
def load_predict(path):
    tok=AutoTokenizer.from_pretrained(path); model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval(); assert model.config.num_labels==2
    def predict(texts,batch=128):
        out=[]
        with torch.inference_mode():
            for i in range(0,len(texts),batch):
                enc=tok(texts[i:i+batch],padding=True,truncation=True,max_length=128,return_tensors='pt').to('cuda')
                out.extend(torch.softmax(model(**enc).logits,dim=-1)[:,1].cpu().tolist())
        return np.array(out)
    return model,predict
sl=lambda key: [i for i,x in enumerate(dev) if str(x['slice'])==key]
fraud=sl('fraud_credentials'); coercive=sl('coercive_control'); ext=sl('digital_extortion')
grooming=[i for i,x in enumerate(dev) if str(x['slice']).startswith('grooming')]
recon=sl('routine_recon')
def evaluate_dev(path):
    model,predict=load_predict(path)
    yr=np.array([int(int(x['label'])>0) for x in real]); pr=predict([x['text'] for x in real])
    yd=np.array([x['label'] for x in dev]); pdv=predict([x['text'] for x in dev])
    pg=predict([x['text'] for x in groom])
    candidates=[]
    for adult in np.arange(.40,.86,.01):
        minor=max(.35,round(float(adult)-.16,2))
        rpred=(pr>=adult)
        dthr=np.array([minor if x['receiver_is_minor'] else adult for x in dev]); dpred=(pdv>=dthr)
        tn,fp,fn,tp=confusion_matrix(yr,rpred,labels=[0,1]).ravel(); dtn,dfp,dfn,dtp=confusion_matrix(yd,dpred,labels=[0,1]).ravel()
        row={'adult':round(float(adult),2),'minor':minor,'real_recall':tp/(tp+fn),'real_specificity':tn/(tn+fp),
             'dev_recall':dtp/(dtp+dfn),'dev_specificity':dtn/(dtn+dfp),
             'fraud_recall':float(dpred[fraud].mean()) if fraud else 1.0,'coercive_recall':float(dpred[coercive].mean()) if coercive else 1.0,
             'grooming_recall':float(dpred[grooming].mean()) if grooming else 1.0,'extortion_recall':float(dpred[ext].mean()) if ext else 1.0,
             'recon_recall':float(dpred[recon].mean()) if recon else 1.0,'synthetic_grooming_recall':float((pg>=minor).mean())}
        row['pass']=all([row['real_recall']>=.80,row['real_specificity']>=.80,row['dev_recall']>=.85,row['dev_specificity']>=.90,
                         row['fraud_recall']>=.80,row['coercive_recall']>=.80,row['grooming_recall']>=.80,row['extortion_recall']>=.80])
        candidates.append(row)
    passing=[x for x in candidates if x['pass']]
    best=max(passing,key=lambda x:(x['real_specificity'],x['dev_specificity'])) if passing else max(candidates,key=lambda x:(min(x['dev_recall'],x['dev_specificity'],x['fraud_recall'],x['coercive_recall']),x['real_specificity']))
    del model; torch.cuda.empty_cache(); return best

In [ ]:
# 5. v6 repeat 1→2→3 재학습. 개발 게이트 통과 시 즉시 중단
ATTEMPTS=[]; SELECTED=None
base=yaml.safe_load((REPO/'configs/module1_binary_hardcases_v6.yaml').read_text())
for repeat in [1,2,3]:
    cfg=json.loads(json.dumps(base)); name=f'module1_binary_hardcases_v6_r{repeat}'
    cfg['data']['extra_train_repeat']=repeat; cfg['model']['checkpoint_dir']=f'models/checkpoints/{name}'; cfg['paths']['checkpoint_dir']=f'models/checkpoints/{name}'
    temp=Path(f'/content/{name}.yaml'); temp.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))
    subprocess.run([sys.executable,'scripts/train_module1.py','--config',str(temp)],cwd=REPO,check=True)
    ckpt=REPO/cfg['model']['checkpoint_dir']; result=evaluate_dev(ckpt); result.update({'repeat':repeat,'checkpoint':str(ckpt)}); ATTEMPTS.append(result); print(json.dumps(result,ensure_ascii=False,indent=2))
    if result['pass']: SELECTED=result; break
report=REPO/'reports/validation_reports/module1_binary_hardcases_v6/dev_attempts.json'; report.parent.mkdir(parents=True,exist_ok=True); report.write_text(json.dumps(ATTEMPTS,ensure_ascii=False,indent=2))
assert SELECTED is not None, '3회 모두 개발 게이트 실패 — blind v8 실행 및 HF 업로드 금지'
print('SELECTED',SELECTED)

In [ ]:
# 6. 후보 고정 후 fresh blind v8 최초 1회 평가 (규칙 보조 OFF)
blind_path=REPO/'tests/fixtures/safe_blind_v8.jsonl'; before=hashlib.sha256(blind_path.read_bytes()).hexdigest()
out=REPO/'artifacts/safe_blind_v8_results.json'
subprocess.run([sys.executable,'scripts/evaluate_safe_blind.py','--model',SELECTED['checkpoint'],'--data',str(blind_path),
                '--adult-threshold',str(SELECTED['adult']),'--minor-threshold',str(SELECTED['minor']),'--no-rule-assist','--output',str(out)],cwd=REPO,check=True)
assert hashlib.sha256(blind_path.read_bytes()).hexdigest()==before, 'blind v8 원문이 변경됨 — 무효'
BLIND=json.loads(out.read_text()); m=BLIND['overall']
risk_slices=[s for s in BLIND['by_slice'] if BLIND['by_slice'][s].get('risk_recall') is not None]
slice_r={s:BLIND['by_slice'][s]['risk_recall'] for s in risk_slices}
BLIND_PASS=bool(m['risk_recall']>=.80 and m['specificity']>=.90 and (min(slice_r.values()) if slice_r else 1.0)>=.75)
print({'blind_pass':BLIND_PASS,'overall':m,'risk_slices':slice_r,'sha256':before})
assert BLIND_PASS, 'blind v8 실패 — 업로드 금지, v8은 회귀셋으로 격하하고 다음 라운드 준비'

In [ ]:
import json
res = json.loads((REPO/'artifacts/safe_blind_v8_results.json').read_text())
for e in res['errors']:
    kind = 'FN' if e['label']==1 else 'FP'
    print(kind, e['slice'], round(e['risk_prob'],4), '|', e['text'])

In [ ]:
# 7. 명시적으로 켠 경우에만 HF 업로드 (추론 파일만; 학습 상태 자동 제외)
UPLOAD_TO_HF=False
if UPLOAD_TO_HF:
    from getpass import getpass
    from huggingface_hub import login,upload_folder
    token=getpass('HF write token: '); login(token=token,add_to_git_credential=False); del token
    url=upload_folder(repo_id='soyuncj/thisabled-safety-kcelectra',folder_path=SELECTED['checkpoint'],
        commit_message=f"retrain external(dktc/kmhas/apeach) v6 r{SELECTED['repeat']} blind-v8 approved",
        ignore_patterns=['checkpoint-*','optimizer*','scheduler*','trainer_state*','rng_state*','training_args*'])
    print('HF_COMMIT_URL:',url)
    print('업로드 후: 새 커밋 SHA를 SAFE_MODEL_REVISION으로 서빙에 반영하고 /health revision 확인.')
else:
    print('검증 완료. 업로드는 비활성 상태입니다. (HF 저장소는 현재 공개)')